# 2. Импорты и конфиг

In [1]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

import pandas as pd

# Работа с данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [2]:
# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
# отключаем научную нотацию для удобаства анализа данных
pd.options.display.float_format = '{:,.0f}'.format

# настройки графиков
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MODELS_DIR,
    MLFLOW_BASE_DIR,
    MLFLOW_DIR,
    MLFLOW_DB_PATH,
    AIRFLOW_DIR,
    AIRFLOW_DAGS_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "MLFLOW_BASE_DIR": MLFLOW_BASE_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "MLFLOW_DB_PATH": MLFLOW_DB_PATH,
    "AIRFLOW_DIR": AIRFLOW_DIR,
    "AIRFLOW_DAGS_DIR": AIRFLOW_DAGS_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\nProject paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<22} {rel_path:<40} [{status}]")

PROJECT_ROOT: ecommerce-recsys
src exists: True

Project paths:

DATA_DIR               data                                     [OK]
RAW_DIR                data/raw                                 [OK]
PROCESSED_DIR          data/processed                           [OK]
ARTIFACTS_DIR          artifacts                                [OK]
MODELS_DIR             artifacts/models                         [OK]
MLFLOW_BASE_DIR        mlflow                                   [OK]
MLFLOW_DIR             mlflow/mlruns                            [OK]
MLFLOW_DB_PATH         mlflow/mlflow.db                         [OK]
AIRFLOW_DIR            airflow                                  [OK]
AIRFLOW_DAGS_DIR       airflow/dags                             [OK]
EVENTS_PATH            data/raw/events.csv                      [OK]
CATEGORY_TREE_PATH     data/raw/category_tree.csv               [OK]
ITEM_PROPERTIES_PATH   data/raw/item_properties.csv             [MISSING]


# 3. Загрузка данных

In [4]:
events = pd.read_csv(
    EVENTS_PATH,
    dtype={
        "visitorid": "int32",
        "event": "category",
        "itemid": "int32",
        "transactionid": "float64",
    },
)

category_tree = pd.read_csv(
    f"{RAW_DIR}/category_tree.csv",
    dtype={
        "categoryid": "int32",
        "parentid": "float64",
    },
)

item_props_part1 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part1.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props_part2 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part2.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props = pd.concat([item_props_part1, item_props_part2], ignore_index=True)

print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

Events shape: (2756101, 5)
Category tree shape: (1669, 2)
Item properties shape: (20275902, 4)


# 3. Предобработка данных

На этом этапе:
- выполняется приведение типов;
- timestamp переводится в datetime;
- из item_properties извлекаются ключевые признаки товаров;
- формируется таблица item-level признаков для дальнейшего моделирования.

## 3.1. Приведение времени и типов

In [ ]:
events["timestamp_dt"] = pd.to_datetime(events["timestamp"], unit="ms")
item_props["timestamp_dt"] = pd.to_datetime(item_props["timestamp"], unit="ms")

events["visitorid"] = events["visitorid"].astype("int32")
events["itemid"] = events["itemid"].astype("int32")
events["event"] = events["event"].astype("category")

item_props["itemid"] = item_props["itemid"].astype("int32")
item_props["property"] = item_props["property"].astype("category")

category_tree["categoryid"] = category_tree["categoryid"].astype("int32")
category_tree["parentid"] = pd.to_numeric(
    category_tree["parentid"], errors="coerce"
).astype("float32")

print(events.dtypes)
print(item_props.dtypes)
print(category_tree.dtypes)

## 3.2. Удаление обрезанного хвоста

In [ ]:
CUTOFF_DATE = pd.Timestamp("2015-09-15")

events = events[events["timestamp_dt"] < CUTOFF_DATE].copy()
item_props = item_props[item_props["timestamp_dt"] < CUTOFF_DATE].copy()

print("events shape after cutoff:", events.shape)
print("item_props shape after cutoff:", item_props.shape)

## 3.3. Извлечение available

In [ ]:
available_df = item_props[item_props["property"] == "available"].copy()

available_df["value_num"] = pd.to_numeric(available_df["value"], errors="coerce")
available_df = available_df.dropna(subset=["value_num"])

available_df = available_df.sort_values(["itemid", "timestamp_dt"])
available_df = available_df.groupby("itemid", as_index=False).tail(1)

available_df = available_df[["itemid", "value_num"]].rename(
    columns={"value_num": "available"}
)
available_df["available"] = available_df["available"].astype("int8")

available_df.head()

## 3.4. Извлечение categoryid

In [ ]:
category_df = item_props[item_props["property"] == "categoryid"].copy()

category_df["value_num"] = pd.to_numeric(category_df["value"], errors="coerce")
category_df = category_df.dropna(subset=["value_num"])

category_df["categoryid"] = category_df["value_num"].astype("int32")

category_df = category_df.sort_values(["itemid", "timestamp_dt"])
category_df = category_df.groupby("itemid", as_index=False).tail(1)

category_df = category_df[["itemid", "categoryid"]]

category_df.head()

## 3.5. Агрегация базовых item-features

Сразу делаем простые признаки по свойствам товара.

In [ ]:
item_prop_counts = (
    item_props.groupby("itemid").size().reset_index(name="item_prop_count")
)

item_unique_props = (
    item_props.groupby("itemid")["property"]
    .nunique()
    .reset_index(name="item_unique_prop_count")
)

item_features = (
    item_prop_counts.merge(item_unique_props, on="itemid", how="outer")
    .merge(available_df, on="itemid", how="left")
    .merge(category_df, on="itemid", how="left")
)

item_features["available"] = item_features["available"].fillna(0).astype("int8")

item_features.head()

## 3.6. Присоединение родительской категории

In [ ]:
item_features = item_features.merge(
    category_tree.rename(columns={"parentid": "parent_categoryid"}),
    on="categoryid",
    how="left",
)

item_features.head()

## 3.7. Фильтрация недоступных товаров

На этапе исследования модели можно сохранить оба варианта:

полный каталог
каталог для рекомендаций

In [ ]:
available_items = set(item_features.loc[item_features["available"] == 1, "itemid"])

print("Available items:", len(available_items))
print(
    "Share of available items:",
    len(available_items) / item_features["itemid"].nunique(),
)

## 3.8. Сохраняем промежуточные артефакты

In [ ]:
item_features.to_parquet(PROCESSED_DIR / "item_features_base.parquet", index=False)
events.to_parquet(PROCESSED_DIR / "events_preprocessed.parquet", index=False)